In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sa = "stretaillakeatharva01"
silver = f"abfss://silver@{sa}.dfs.core.windows.net"
gold = f"abfss://gold@{sa}.dfs.core.windows.net"

orders = spark.read.format("delta").load(f"{silver}/orders")
dim = spark.read.format("delta").load(f"{silver}/dim_customers").filter("is_current = true")

def save_gold(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold}/{name}")
    print(name, "->", df.count(), "rows")

In [0]:
daily = (orders.groupBy("order_date", "order_status")
         .agg(F.count("*").alias("order_count"))
         .orderBy("order_date", "order_status"))
save_gold(daily, "daily_orders_by_status")

daily_orders_by_status -> 3203 rows


In [0]:
total = orders.count()
funnel = (orders.groupBy("order_status")
          .agg(F.count("*").alias("order_count"))
          .withColumn("pct_of_total", F.round(F.col("order_count") * 100 / total, 2))
          .orderBy(F.desc("order_count")))
save_gold(funnel, "order_status_funnel")
display(funnel)

order_status_funnel -> 9 rows


order_status,order_count,pct_of_total
COMPLETE,22900,33.24
PENDING_PAYMENT,15031,21.82
PROCESSING,8275,12.01
PENDING,7610,11.05
CLOSED,7558,10.97
ON_HOLD,3798,5.51
SUSPECTED_FRAUD,1558,2.26
CANCELED,1428,2.07
PAYMENT_REVIEW,729,1.06


In [0]:
by_state = (orders.join(dim.select("customer_id", "state", "city"), "customer_id", "left")
            .groupBy("state")
            .agg(F.count("*").alias("total_orders"),
                 F.countDistinct("customer_id").alias("unique_customers"),
                 F.sum(F.when(F.col("order_status") == "COMPLETE", 1).otherwise(0)).alias("complete_orders"))
            .withColumn("complete_pct", F.round(F.col("complete_orders") * 100 / F.col("total_orders"), 2))
            .orderBy(F.desc("total_orders")))
save_gold(by_state, "orders_by_state")
display(by_state.limit(10))

orders_by_state -> 45 rows


state,total_orders,unique_customers,complete_orders,complete_pct
PR,26499,4760,8734,32.96
CA,11046,2005,3683,33.34
NY,4331,772,1451,33.5
TX,3442,633,1160,33.7
IL,2885,523,971,33.66
FL,2089,374,704,33.7
OH,1575,275,523,33.21
PA,1458,261,498,34.16
MI,1428,252,487,34.1
NJ,1259,219,410,32.57


In [0]:
ref_date = orders.agg(F.max("order_date")).collect()[0][0]  # data ki last date, aaj ki nahi

rfm = (orders.groupBy("customer_id")
       .agg(F.max("order_date").alias("last_order_date"),
            F.count("*").alias("frequency"))
       .withColumn("recency_days", F.datediff(F.lit(ref_date), F.col("last_order_date")))
       .withColumn("r_score", F.ntile(4).over(Window.orderBy(F.desc("recency_days"))))
       .withColumn("f_score", F.ntile(4).over(Window.orderBy("frequency")))
       .withColumn("segment",
            F.when((F.col("r_score") >= 3) & (F.col("f_score") >= 3), "Champion")
             .when(F.col("f_score") >= 3, "Loyal")
             .when(F.col("r_score") <= 2, "At Risk")
             .otherwise("Regular"))
       .join(dim.select("customer_id", "customer_fname", "customer_lname", "state"), "customer_id", "left"))

save_gold(rfm, "customer_rfm")
display(rfm.groupBy("segment").count().orderBy(F.desc("count")))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_rfm -> 12405 rows


segment,count
At Risk,3961
Champion,3960
Regular,2242
Loyal,2242


In [0]:
top = rfm.orderBy(F.desc("frequency")).limit(20)
save_gold(top, "top_customers")
display(top)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


top_customers -> 20 rows


customer_id,last_order_date,frequency,recency_days,r_score,f_score,segment,customer_fname,customer_lname,state
5897,2014-07-19,16,5,4,4,Champion,Mary,Griffin,NY
12431,2014-07-14,16,10,4,4,Champion,Mary,Rios,HI
6316,2014-06-26,16,28,3,4,Champion,Kyle,Smith,PR
569,2014-06-17,16,37,3,4,Champion,Mary,Frye,ID
4320,2014-06-04,15,50,2,4,Loyal,Jordan,Taylor,PR
12284,2014-07-06,15,18,4,4,Champion,Mary,Smith,CT
5654,2014-07-12,15,12,4,4,Champion,Jerry,Smith,PR
5283,2014-07-21,15,3,4,4,Champion,Jacob,Guerrero,CA
221,2014-07-18,15,6,4,4,Champion,Mary,Cox,OH
5624,2014-06-13,15,41,3,4,Champion,Mary,Mata,PR


In [0]:
display(dbutils.fs.ls(gold))

path,name,size,modificationTime
abfss://gold@stretaillakeatharva01.dfs.core.windows.net/customer_rfm/,customer_rfm/,0,1792427889000
abfss://gold@stretaillakeatharva01.dfs.core.windows.net/daily_orders_by_status/,daily_orders_by_status/,0,1792427857000
abfss://gold@stretaillakeatharva01.dfs.core.windows.net/order_status_funnel/,order_status_funnel/,0,1792427865000
abfss://gold@stretaillakeatharva01.dfs.core.windows.net/orders_by_state/,orders_by_state/,0,1792427876000
abfss://gold@stretaillakeatharva01.dfs.core.windows.net/top_customers/,top_customers/,0,1792427904000
